# 1. Archiving

**RadDB** turns radar volumes into a compact, queryable Parquet archive. A volume
is an [xarray](https://docs.xarray.dev/) `DataTree` (one group per sweep).

This notebook covers:

1. Raw data and RadDB initialisation
2. Archiving
3. Archived data

---
## How the archive is stored

A radar data is stored as **static data (LUT)** (per-gate geometry, computed once) plus
**dynamic data, one file per volume** (polarimetric variables), linked by an integer `gate_id`.
Following is an example of one archived volume (paths and files):

```
{archive_dir}/{radar}/LUT/{radar}_LUT.parquet          # gate centroids
{archive_dir}/{radar}/LUT/{radar}_h_plane_LUT.parquet  # horizontal gate plane (for PPI)
{archive_dir}/{radar}/LUT/{radar}_v_plane_LUT.parquet  # vertical gate plane  (for RHI)
{archive_dir}/{radar}/LUT/{radar}_corners_LUT.parquet  # 3-D gate corners
{archive_dir}/{radar}/LUT/{radar}_info.yaml            # site, CRS, scan geometry

{archive_dir}/{radar}/{YYYY}/{MM}/{DD}/{radar}_{YYYYMMDD}_{HHMMSS}_POL.parquet    # dynamic data
```

The geometry is stored **once**, not once per volume — which is what keeps the
archive small. Gates with no echo are dropped at archive time (`DBZH > 0` by
default).

In [ ]:
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path

import raddb
from raddb.lut import suggest_crs

print("raddb", raddb.__version__)

raddb 0.1.dev5+gde6070734.d20260323


## 1. Raw data and RadDB initialisation

### Input paths

`FMI_DIR` and `NEXRAD_DIR` hold the DataTree volumes to be archived; `ARCHIVE_DIR`
is where RadDB writes the archive. Edit them to match your own machine.

In [ ]:
# --------------------------------------------------------------------------
# CONFIGURATION — point these at your own data
# --------------------------------------------------------------------------
# Any xarray DataTree with the standard xradar layout works.
# These tutorials use Finnish (FMI) and US (NEXRAD) volumes stored as zarr and nc format.
# Both networks publish openly, so every example here can be reproduced.
# Edit the paths below to point at your own data.

FMI_DIR = Path("~/Desktop/LTE_project/ltenas8/data/RADAR/FMI_datatree_zarr").expanduser()
NEXRAD_DIR = Path("~/Desktop/LTE_project/ltenas8/data/RADAR/NEXRAD_datatree_zarr").expanduser()
ARCHIVE_DIR = Path("~/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive").expanduser()

print("FMI DataTrees   :", FMI_DIR)
print("NEXRAD DataTrees:", NEXRAD_DIR)
print("Archive         :", ARCHIVE_DIR)

FMI DataTrees   : /home/erik_poschivo/Desktop/LTE_project/ltenas8/data/RADAR/FMI_datatree_zarr
NEXRAD DataTrees: /home/erik_poschivo/Desktop/LTE_project/ltenas8/data/RADAR/NEXRAD_datatree_zarr
Archive         : /home/erik_poschivo/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive


### Inspecting raw data archive

`inventory(datatree_dir=...)` scans a directory of DataTree files and prints what
it finds: the radar name taken from each filename prefix, the number of files, the
time span they cover and their total size on disk.

In [ ]:
db = raddb.RadDB()
db.inventory(datatree_dir=NEXRAD_DIR)
# db.inventory(datatree_dir=FMI_DIR)

RadDB inventory — DataTree files on disk (not archived yet)
  directory : /home/erik_poschivo/Desktop/LTE_project/ltenas8/data/RADAR/NEXRAD_datatree_zarr
  files     : 107
  radars    : KLOT, KMLB, KTLX  (from the filename prefix)
  time range: 2024-06-12 22:00:00 .. 2024-08-18 18:59:21
------------------------------------------------------------------------------
  radar     files  time range                                           size


  KLOT         40  2024-06-12 22:01:28 .. 2024-08-18 18:57:49       366.2 MB


  KMLB         35  2024-06-12 22:00:00 .. 2024-08-18 18:56:53       450.9 MB


  KTLX         32  2024-06-12 22:03:24 .. 2024-08-18 18:59:21       440.9 MB
------------------------------------------------------------------------------
  archive with: db.archive(datatree_dir='/home/erik_poschivo/Desktop/LTE_project/ltenas8/data/RADAR/NEXRAD_datatree_zarr')


In [ ]:
# `detailed=True` adds a per-day breakdown
db.inventory(datatree_dir=NEXRAD_DIR, detailed=True)
# db.inventory(datatree_dir=FMI_DIR, detailed=True)

RadDB inventory — DataTree files on disk (not archived yet)
  directory : /home/erik_poschivo/Desktop/LTE_project/ltenas8/data/RADAR/NEXRAD_datatree_zarr
  files     : 107
  radars    : KLOT, KMLB, KTLX  (from the filename prefix)
  time range: 2024-06-12 22:00:00 .. 2024-08-18 18:59:21
------------------------------------------------------------------------------
  radar     files  time range                                           size


  KLOT         40  2024-06-12 22:01:28 .. 2024-08-18 18:57:49       366.2 MB
      2024-06-12     15 volume(s)  22:01:28 .. 22:58:56
      2024-07-05     13 volume(s)  22:04:13 .. 22:58:08
      2024-08-18     12 volume(s)  18:06:21 .. 18:57:49


  KMLB         35  2024-06-12 22:00:00 .. 2024-08-18 18:56:53       450.9 MB
      2024-06-12     13 volume(s)  22:00:00 .. 22:57:27
      2024-07-05     13 volume(s)  22:00:13 .. 22:57:30
      2024-08-18      9 volume(s)  18:01:11 .. 18:56:53


  KTLX         32  2024-06-12 22:03:24 .. 2024-08-18 18:59:21       440.9 MB
      2024-06-12      8 volume(s)  22:03:24 .. 22:53:30
      2024-07-05     15 volume(s)  22:02:39 .. 22:57:52
      2024-08-18      9 volume(s)  18:03:18 .. 18:59:21
------------------------------------------------------------------------------
  archive with: db.archive(datatree_dir='/home/erik_poschivo/Desktop/LTE_project/ltenas8/data/RADAR/NEXRAD_datatree_zarr')


### Creating the RadDB object

`RadDB(archive_dir=..., crs=...)` returns an *archive-bound* RadDB: it knows where
the archive lives and which projection to write it in. This is the object used to
archive, open and inspect data.

In [ ]:
db = raddb.RadDB(archive_dir=ARCHIVE_DIR, crs=3067)  # 3067 ==> ETRS89 / TM35FIN (all of Finland)
db

RadDB(archive_dir=/home/erik_poschivo/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive, crs=3067) [archive-bound, no data loaded]

In [ ]:
db_us = raddb.RadDB(archive_dir=ARCHIVE_DIR)
db_us

RadDB(archive_dir=/home/erik_poschivo/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive, crs=None) [archive-bound, no data loaded]

## 2. Archiving

`archive()` takes either a directory of DataTree files or an in-memory DataTree.
The LUT is generated automatically from the first volume of each radar.

In [ ]:
result = db.archive(datatree_dir=FMI_DIR, time_period=("2024-06-01", "2024-06-15"))
result

RadDB archive
  archive_dir : /home/erik_poschivo/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive
  crs         : 3067
  radars      : ['FANJ', 'FKOR', 'FKUO']
  filter      : keep DBZH > 0.0
  volumes     : 41 archived, 0 failed
  elapsed     : 1m 35s


{'n_archived': 41,
 'n_failed': 0,
 'n_skipped': 0,
 'radars': ['FANJ', 'FKOR', 'FKUO']}

### The CRS constraint

**A projection is mandatory to write an archive, and never needed to read one. Can be given when RadDB is initilaized or at archiving time.**

The LUT stores projected gate coordinates, and every crop and cross-section is
computed in them. A wrong projection is therefore silently wrong: EPSG:3067 (Finnish
TM35FIN) used outside Finland mis-measures distance, while the
results could still look perfectly normal. RadDB has no default, the CRS is stated once,
when the object is created, and is checked against the radar's real position before
anything is written.

---

#### Example for Finland:

All Finnish radars fit one national projection, so the three FMI radars above were
archived together with `crs=3067`. That is the exception, not the rule — it works
because Finland is narrow enough for a single transverse Mercator zone.

---

#### Example for US:
A projection is only valid for one large region, so for US radars is chosen per radar . `KTLX` sits in
UTM zone 14N; `KLOT` and `KMLB` are in zones 16N and 17N and would be refused with
that CRS.

In [ ]:
# check what's the suggested crs of the 3 US radars in the NEXRAD dataset
# "KTLX": lat/lon = 35.333/-97.278
# "KMLB": lat/lon = 28.113/-80.654
# "KLOT": lat/lon = 41.604/-88.084
print(f"KTLX ==> {suggest_crs(latitude=35.333, longitude=-97.278)}")
print(f"KMLB ==> {suggest_crs(latitude=28.113, longitude=-80.654)}")
print(f"KLOT ==> {suggest_crs(latitude=41.604, longitude=-88.084)}")

KTLX ==> 32614
KMLB ==> 32617
KLOT ==> 32616


In [ ]:
db_us.archive(
    datatree_dir=NEXRAD_DIR,
    radar=["KTLX"],
    crs=32614,  # 32614 ==> UTM zone 14N
    time_period=("2024-01-01", "2024-06-15"),
)
db_us.archive(
    datatree_dir=NEXRAD_DIR,
    radar=["KMLB"],
    crs=32617,  # 32617 ==> UTM zone 17N
    time_period=("2024-01-01", "2024-06-15"),
)
db_us.archive(
    datatree_dir=NEXRAD_DIR,
    radar=["KLOT"],
    crs=32616,  # 32616 ==> UTM zone 16N
    time_period=("2024-01-01", "2024-06-15"),
)

RadDB archive
  archive_dir : /home/erik_poschivo/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive
  crs         : 32614
  radars      : ['KTLX']
  filter      : keep DBZH > 0.0
  volumes     : 0 archived, 0 failed
  elapsed     : 0s


  [KMLB] FAIL KMLB_20240612_221403: radar 'KMLB': the LUT has no sweep 17, but this volume does — it uses a different scan strategy.


  [KMLB] FAIL KMLB_20240612_222407: radar 'KMLB' sweep 7: volume has 720 rays, the LUT was built for 360 — a different scan strategy. Archive it under its own radar name, or rebuild the LUT.


  [KMLB] FAIL KMLB_20240612_223344: radar 'KMLB' sweep 7: volume has 720 rays, the LUT was built for 360 — a different scan strategy. Archive it under its own radar name, or rebuild the LUT.


  [KMLB] FAIL KMLB_20240612_223820: radar 'KMLB' sweep 7: volume has 720 rays, the LUT was built for 360 — a different scan strategy. Archive it under its own radar name, or rebuild the LUT.
RadDB archive
  archive_dir : /home/erik_poschivo/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive
  crs         : 32617
  radars      : ['KMLB']
  filter      : keep DBZH > 0.0
  volumes     : 0 archived, 4 failed
  elapsed     : 40s
RadDB archive
  archive_dir : /home/erik_poschivo/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive
  crs         : 32616
  radars      : ['KLOT']
  filter      : keep DBZH > 0.0
  volumes     : 0 archived, 0 failed
  elapsed     : 0s


{'n_archived': 0, 'n_failed': 0, 'n_skipped': 0, 'radars': ['KLOT']}

## 3. Archived data

In [ ]:
db = raddb.RadDB(archive_dir=ARCHIVE_DIR)
print("radars in the archive:", db.list_radars())
db.inventory()

radars in the archive: ['FANJ', 'FKOR', 'FKUO', 'KDVN', 'KLOT', 'KMLB', 'KTLX']
RadDB inventory — archived data
  archive_dir : /home/erik_poschivo/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive
  radars      : FANJ, FKOR, FKUO, KDVN, KLOT, KMLB, KTLX
  volumes     : 51
  time range  : 2024-06-01 12:00:01 .. 2024-06-25 23:04:58
------------------------------------------------------------------------------
  radar   volumes  time range                                           size
  FANJ         13  2024-06-01 12:00:02 .. 2024-06-14 12:00:03        29.3 MB
  FKOR         14  2024-06-01 12:00:01 .. 2024-06-14 12:00:01        13.3 MB
  FKUO         14  2024-06-01 12:00:05 .. 2024-06-14 12:00:05        25.9 MB
  KDVN          1  2024-06-25 23:04:58                               17.6 MB
  KLOT          0  unknown (no timestamp in the filenames)               0 B
  KMLB          1  2024-06-12 22:57:27                                7.1 MB
  KTLX          8  2024-06-12 22:

In [ ]:
for p in sorted((ARCHIVE_DIR / "FANJ" / "LUT").iterdir()):
    print(f"  {p.name:<28} {p.stat().st_size / 1e6:8.2f} MB")

  FANJ_LUT.parquet                65.90 MB
  FANJ_corners_LUT.parquet        20.74 MB
  FANJ_h_plane_LUT.parquet        20.30 MB
  FANJ_info.yaml                   0.00 MB
  FANJ_v_plane_LUT.parquet         0.45 MB


### `gate_id`: how a volume finds its geometry

One int64 per gate links a row of data (polarimetric variables) to its row of geometry:

```
gate_id = radar_code * 10^12 + sweep * 10^10 + azimuth*10 * 10^6 + range_m
```

In [ ]:
lut = db.get_lut("FANJ")
print("\nLUT:", lut.shape)
print(lut.columns)
print(lut.head(5).select(["gate_id", "sweep", "azimuth", "range", "latitude", "longitude", "altitude"]))
print(lut.head(5).select(["x", "y", "z", "x_3067", "y_3067"]))


LUT: (1610280, 13)
['gate_id', 'sweep', 'azimuth', 'range', 'elevation_angle', 'latitude', 'longitude', 'altitude', 'x', 'y', 'z', 'x_3067', 'y_3067']
shape: (5, 7)
┌────────────────────┬───────┬─────────┬────────┬───────────┬───────────┬────────────┐
│ gate_id            ┆ sweep ┆ azimuth ┆ range  ┆ latitude  ┆ longitude ┆ altitude   │
│ ---                ┆ ---   ┆ ---     ┆ ---    ┆ ---       ┆ ---       ┆ ---        │
│ i64                ┆ i32   ┆ f64     ┆ f32    ┆ f64       ┆ f64       ┆ f64        │
╞════════════════════╪═══════╪═════════╪════════╪═══════════╪═══════════╪════════════╡
│ 713647000000000250 ┆ 0     ┆ 0.0     ┆ 250.0  ┆ 60.906118 ┆ 27.10806  ┆ 140.31267  │
│ 713647000000000750 ┆ 0     ┆ 0.0     ┆ 750.0  ┆ 60.910615 ┆ 27.10806  ┆ 142.960081 │
│ 713647000000001250 ┆ 0     ┆ 0.0     ┆ 1250.0 ┆ 60.915111 ┆ 27.10806  ┆ 145.636922 │
│ 713647000000001750 ┆ 0     ┆ 0.0     ┆ 1750.0 ┆ 60.919608 ┆ 27.10806  ┆ 148.343192 │
│ 713647000000002250 ┆ 0     ┆ 0.0     ┆ 2250.0 ┆ 6

### Radar site metadata

`info.yaml` records everything needed to reconstruct the geometry — including the
CRS that was validated at archive time, and the radar's **scan strategy**.

In [ ]:
info = db.get_radar_info("FANJ")
for k in ["radar", "network", "latitude", "longitude", "altitude", "crs", "ke", "beamwidth_deg", "n_sweeps", "n_gates"]:
    print(f"  {k:<16} {info[k]}")

  radar            FANJ
  network          
  latitude         60.90387001633644
  longitude        27.1080600656569
  altitude         139.0
  crs              {'epsg': 3067, 'columns': ['x_3067', 'y_3067']}
  ke               1.3333333333333333
  beamwidth_deg    1.0
  n_sweeps         13
  n_gates          1610280


---
**Next:** [2 — Opening and filtering](02_opening_and_filtering.ipynb)